# Machine Learning Final Project
## Network Intrusion Detection — NSL-KDD Dataset

**Team (CS-2406):** Nurzhan Bekmurat · Shavkat Khanezar · Yernur Khuan  
**Course:** Machine Learning (Advanced Level) — AMO 3208  
**Instructor:** Aigul B. Mimenbayeva  
**University:** Astana IT University  

**Task:** Binary Classification — Normal Traffic (0) vs. Attack Traffic (1)  
**Models (5):** Logistic Regression · Random Forest · K-Nearest Neighbors · XGBoost · Support Vector Machine  
**Tuning:** GridSearchCV on the top-2 baseline models  
**Metrics:** Accuracy · Precision · Recall · F1-score · ROC-AUC · 5-fold CV F1  
**Deployment:** Best model is serialized for a Streamlit web application (`app.py`).

---
## 1. Project Setup

**Objective.** Build a complete ML pipeline that classifies a network connection as *normal* or *attack* using 41 NSL-KDD connection-level features. The pipeline covers data loading, preprocessing, feature engineering, training and comparison of five algorithms, hyperparameter tuning, model serialization, and a Streamlit application for real-time inference.

**Why NSL-KDD.** NSL-KDD is the de-facto academic benchmark for intrusion-detection research. It fixes the duplicate and class-balance issues of the original KDD-99 set and is maintained by the Canadian Institute for Cybersecurity at the University of New Brunswick.

**Why these five models.** They span the main families used in IDS literature:
* Logistic Regression — linear, interpretable baseline.
* Random Forest — bagging ensemble of trees, scale-invariant, gives feature importances.
* KNN — non-parametric, distance-based.
* XGBoost — gradient-boosted trees, current state-of-the-art on tabular data.
* SVM (linear kernel) — margin-based classifier, complementary decision principle to trees.

In [ ]:

import os, urllib.request, joblib, warnings, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix, roc_curve
)
from xgboost import XGBClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print('Libraries imported.')

---
## 2. Data Collection

**Source.** Public mirror of the official NSL-KDD training set (`KDDTrain+.txt`) from the `defcom17/NSL_KDD` GitHub repository. We download directly via `urllib`, so the notebook is reproducible without Google Colab uploads.  
**Size.** 125 973 rows × 43 columns.  
**Variables.** 38 numeric + 3 categorical (`protocol_type`, `service`, `flag`) + `attack_type` (string) + `difficulty` (integer). The file has *no header row*; column names from UNB documentation are assigned in code.

In [ ]:
URL = 'https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTrain%2B.txt'
LOCAL = 'KDDTrain+.txt'

if not os.path.exists(LOCAL):
    print('Downloading NSL-KDD training set ...')
    urllib.request.urlretrieve(URL, LOCAL)
print(f'File ready: {LOCAL} ({os.path.getsize(LOCAL)/1e6:.2f} MB)')

col_names = [
    'duration','protocol_type','service','flag','src_bytes','dst_bytes',
    'land','wrong_fragment','urgent','hot','num_failed_logins','logged_in',
    'num_compromised','root_shell','su_attempted','num_root','num_file_creations',
    'num_shells','num_access_files','num_outbound_cmds','is_host_login',
    'is_guest_login','count','srv_count','serror_rate','srv_serror_rate',
    'rerror_rate','srv_rerror_rate','same_srv_rate','diff_srv_rate',
    'srv_diff_host_rate','dst_host_count','dst_host_srv_count',
    'dst_host_same_srv_rate','dst_host_diff_srv_rate','dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate','dst_host_serror_rate','dst_host_srv_serror_rate',
    'dst_host_rerror_rate','dst_host_srv_rerror_rate',
    'attack_type','difficulty'
]

df_raw = pd.read_csv(LOCAL, header=None, names=col_names)
print(f'Loaded: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')
df_raw.head(3)

In [ ]:
# Binary label: normal = 0, any attack = 1
df_raw['label'] = df_raw['attack_type'].apply(lambda x: 0 if str(x).strip()=='normal' else 1)
print('Class distribution:')
print(df_raw['label'].value_counts())
print(f'Attack rate: {df_raw["label"].mean()*100:.1f}%')
print('\nTop attack categories:')
print(df_raw[df_raw['label']==1]['attack_type'].value_counts().head(10))

---
## 3. Data Preparation

Steps applied:
1. Drop `attack_type` (string target) and `difficulty` (dataset artifact, not predictive).
2. Label-encode the three categorical columns (`protocol_type`, `service`, `flag`). Each encoder is kept for re-use in the Streamlit app.
3. Confirm no missing values (a `fillna(median)` is included as a safeguard).
4. Stratified 80/20 train-test split to preserve the ~46.5 % attack ratio.
5. StandardScaler (zero-mean, unit-variance) fitted **on the training set only**, then applied to both. Used by Logistic Regression, KNN, and SVM; tree-based models receive the raw data.

In [ ]:
df = df_raw.copy().drop(columns=['attack_type', 'difficulty'])

cat_cols = ['protocol_type', 'service', 'flag']
encoders = {}
for col in cat_cols:
    enc = LabelEncoder()
    df[col] = enc.fit_transform(df[col].astype(str))
    encoders[col] = enc          # keep for the Streamlit app

df = df.fillna(df.median(numeric_only=True))

print(f'Missing values after fillna: {df.isnull().sum().sum()}')
print(f'Categorical encoders saved: {list(encoders.keys())}')

---
## 4. Feature Engineering

Two domain-informed features are added on top of the 41 original NSL-KDD attributes:

| Feature | Formula | Rationale |
|---|---|---|
| `bytes_ratio` | `src_bytes / (dst_bytes + 1)` | Traffic asymmetry — attacks (esp. DoS, R2L) often push much more data than they receive. |
| `error_rate_total` | `serror_rate + rerror_rate` | Aggregated TCP error signal — port-scan and DoS attacks generate high error rates. |

After engineering: **43 features** total.

In [ ]:
# Correlation analysis (before engineering)
corr = df.corr()['label'].drop('label').abs().sort_values(ascending=False)
corr.head(15).plot(kind='bar', figsize=(10,4), color='#2980b9', alpha=0.85)
plt.title('Top 15 features by |Pearson correlation| with label', fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('fig_correlation.png', dpi=120, bbox_inches='tight')
plt.show()
print(corr.head(10).round(3))

In [ ]:
df['bytes_ratio']      = df['src_bytes'] / (df['dst_bytes'] + 1)
df['error_rate_total'] = df['serror_rate'] + df['rerror_rate']

X = df.drop('label', axis=1)
y = df['label']
feature_names = X.columns.tolist()
print(f'Final feature count: {X.shape[1]} ({X.shape[1]-2} original + 2 engineered)')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}')

In [ ]:
# Visualise class balance + top attack categories
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
counts = y.value_counts()
axes[0].bar(['Normal (0)', 'Attack (1)'], counts.values,
            color=['#27ae60','#e74c3c'], edgecolor='white')
axes[0].set_title('Class distribution', fontweight='bold')
for i, v in enumerate(counts.values):
    axes[0].text(i, v+200, f'{v:,}', ha='center', fontweight='bold')

atk = df_raw[df_raw['label']==1]['attack_type'].value_counts().head(10)
atk.plot(kind='barh', ax=axes[1], color='#c0392b', alpha=0.8)
axes[1].set_title('Top-10 attack categories', fontweight='bold')
axes[1].invert_yaxis()
plt.tight_layout()
plt.savefig('fig_eda.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 5. Model Training & Evaluation — Five Algorithms

Each model is trained on the train split and evaluated on the held-out test split with the same six metrics. A 5-fold stratified CV F1-score is also reported to gauge generalization.

### 5.1 Mathematical formulations (summary)

* **Logistic Regression.** `P(y=1|x) = σ(wᵀx + b)` with `σ(z)=1/(1+e^-z)`; loss = binary cross-entropy.
* **Random Forest.** Majority vote over `B=100` decision trees, each grown on a bootstrap sample with random `√p` features per split. Each split minimizes Gini impurity `G = 1 − Σ pₖ²`.
* **KNN.** Predict class by majority of `k=5` nearest points under Euclidean distance.
* **XGBoost.** Additive boosted trees `F_m(x) = F_{m-1}(x) + η·h_m(x)` minimizing logistic loss with L1/L2 regularization on leaf weights.
* **SVM (linear).** Solve `min ½‖w‖² + CΣξᵢ` s.t. `yᵢ(wᵀxᵢ+b) ≥ 1−ξᵢ`. Probabilities obtained via Platt-style calibration.

In [ ]:
def evaluate(name, model, X_tr, X_te, scaled, do_cv=True):
    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1] if hasattr(model, 'predict_proba') else None
    cv = cross_val_score(model, X_tr, y_train, cv=5, scoring='f1', n_jobs=-1).mean() if do_cv else np.nan
    return {
        'Model':     name,
        'Accuracy':  accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall':    recall_score(y_test, y_pred, zero_division=0),
        'F1':        f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC':   roc_auc_score(y_test, y_prob) if y_prob is not None else np.nan,
        'CV F1':     cv,
        '_model':    model, '_prob': y_prob, '_scaled': scaled, '_pred': y_pred
    }

results = []
print('Training 5 models ...')

results.append(evaluate('Logistic Regression',
    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    X_train_sc, X_test_sc, scaled=True))

results.append(evaluate('Random Forest',
    RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
    X_train, X_test, scaled=False))

results.append(evaluate('KNN',
    KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    X_train_sc, X_test_sc, scaled=True))

results.append(evaluate('XGBoost',
    XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                  subsample=0.9, colsample_bytree=0.9,
                  eval_metric='logloss', random_state=RANDOM_STATE,
                  n_jobs=-1, tree_method='hist'),
    X_train, X_test, scaled=False))

# LinearSVC is fast but has no predict_proba — wrap in CalibratedClassifierCV
svm_base = LinearSVC(C=1.0, max_iter=4000, dual=False, random_state=RANDOM_STATE)
svm_cal  = CalibratedClassifierCV(svm_base, cv=3)
results.append(evaluate('SVM (linear)', svm_cal,
    X_train_sc, X_test_sc, scaled=True, do_cv=False))   # CV is slow for SVM, skip

print('Done.')

In [ ]:
metric_cols = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC', 'CV F1']
cmp_df = pd.DataFrame([{k: r[k] for k in ['Model'] + metric_cols} for r in results]).set_index('Model')
print('===== MODEL COMPARISON (test set) =====')
print(cmp_df.round(4).to_string())
cmp_df.to_csv('results_table.csv')

In [ ]:
# Grouped-bar metric comparison + ROC curves
colors = ['#3498db', '#27ae60', '#e67e22', '#8e44ad', '#16a085']
metrics_plot = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']
x = np.arange(len(metrics_plot))
w = 0.16

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for i, r in enumerate(results):
    vals = [r[m] if not pd.isna(r[m]) else 0 for m in metrics_plot]
    axes[0].bar(x + i*w, vals, w, label=r['Model'], color=colors[i], alpha=0.9)
axes[0].set_xticks(x + 2*w)
axes[0].set_xticklabels(metrics_plot)
axes[0].set_ylim(0.85, 1.02)
axes[0].legend(loc='lower right', fontsize=9)
axes[0].grid(axis='y', alpha=0.3)
axes[0].set_title('Model performance comparison (5 algorithms)', fontweight='bold')

for r, c in zip(results, colors):
    if r['_prob'] is not None:
        fpr, tpr, _ = roc_curve(y_test, r['_prob'])
        axes[1].plot(fpr, tpr, color=c, lw=2, label=f"{r['Model']} (AUC={r['ROC-AUC']:.3f})")
axes[1].plot([0,1], [0,1], 'k--', alpha=0.4)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC curves', fontweight='bold')
axes[1].legend(loc='lower right', fontsize=9)
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig('fig_model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Confusion matrices, 1x5
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, r in zip(axes, results):
    cm = confusion_matrix(y_test, r['_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Normal','Attack'], yticklabels=['Normal','Attack'], cbar=False)
    ax.set_title(r['Model'], fontweight='bold', fontsize=11)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig('fig_confusion.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 6. Hyperparameter Tuning — Top-2 Candidates

The two best baseline models (by F1) are tuned with `GridSearchCV` (5-fold, F1-scoring). Tuning two models — instead of just one — lets us confirm the ranking after optimization and pick a more robust deployment model.

In [ ]:
# --- Random Forest tuning ---
rf_grid = {
    'n_estimators':      [100, 200],
    'max_depth':         [10, 20, None],
    'max_features':      ['sqrt', 'log2'],
    'min_samples_split': [2, 5]
}
print('Tuning Random Forest ... (~2–4 min)')
gs_rf = GridSearchCV(RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
                     rf_grid, cv=5, scoring='f1', n_jobs=-1, verbose=1)
gs_rf.fit(X_train, y_train)
print('Best RF params:', gs_rf.best_params_)
print(f'Best RF CV F1: {gs_rf.best_score_:.4f}')

In [ ]:
# --- XGBoost tuning ---
xgb_grid = {
    'n_estimators':     [200, 400],
    'max_depth':        [4, 6, 8],
    'learning_rate':    [0.05, 0.1],
    'subsample':        [0.8, 1.0]
}
print('Tuning XGBoost ... (~2–5 min)')
gs_xgb = GridSearchCV(
    XGBClassifier(eval_metric='logloss', random_state=RANDOM_STATE,
                  n_jobs=-1, tree_method='hist'),
    xgb_grid, cv=5, scoring='f1', n_jobs=-1, verbose=1)
gs_xgb.fit(X_train, y_train)
print('Best XGB params:', gs_xgb.best_params_)
print(f'Best XGB CV F1: {gs_xgb.best_score_:.4f}')

In [ ]:
# Compare tuned models on test set
def quick_eval(model, X_te):
    yp = model.predict(X_te); yprob = model.predict_proba(X_te)[:, 1]
    return dict(
        Accuracy =accuracy_score(y_test, yp),
        Precision=precision_score(y_test, yp),
        Recall   =recall_score(y_test, yp),
        F1       =f1_score(y_test, yp),
        AUC      =roc_auc_score(y_test, yprob))

tuned_rf  = gs_rf.best_estimator_
tuned_xgb = gs_xgb.best_estimator_
rf_eval  = quick_eval(tuned_rf,  X_test)
xgb_eval = quick_eval(tuned_xgb, X_test)

tuned_table = pd.DataFrame({'RF (tuned)': rf_eval, 'XGB (tuned)': xgb_eval}).round(4)
print('===== TUNED MODELS (test set) =====')
print(tuned_table.to_string())

In [ ]:
# Feature importances of the chosen deployment model
best_model = tuned_rf if rf_eval['F1'] >= xgb_eval['F1'] else tuned_xgb
best_name  = 'RandomForest (tuned)' if best_model is tuned_rf else 'XGBoost (tuned)'
print(f'>>> Selected for deployment: {best_name}')

fi = pd.Series(best_model.feature_importances_, index=feature_names).sort_values(ascending=False).head(15)
fi.plot(kind='barh', figsize=(9, 5), color='#2980b9', alpha=0.85)
plt.title(f'Top 15 feature importances — {best_name}', fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('fig_feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 7. Final Results Summary

In [ ]:
tuned_rows = [
    {'Model': 'Random Forest (tuned)', **rf_eval,  'CV F1': gs_rf.best_score_},
    {'Model': 'XGBoost (tuned)',       **xgb_eval, 'CV F1': gs_xgb.best_score_}
]
# rename AUC -> ROC-AUC to match baseline table
for r in tuned_rows:
    r['ROC-AUC'] = r.pop('AUC')

all_rows = [{k: r[k] for k in ['Model'] + metric_cols} for r in results] + tuned_rows
final_df = pd.DataFrame(all_rows).set_index('Model').round(4)
print('===== FINAL RESULTS — 5 baselines + 2 tuned =====')
print(final_df.to_string())
final_df.to_csv('results_table_final.csv')

---
## 8. Save Artifacts for the Streamlit Application

We serialize the deployment model together with the StandardScaler, the categorical LabelEncoders, and the ordered feature list. `app.py` loads these artifacts and exposes a web UI that accepts either manual feature values or a CSV upload.

In [ ]:
ART_DIR = 'artifacts'
os.makedirs(ART_DIR, exist_ok=True)

joblib.dump(best_model, f'{ART_DIR}/best_model.pkl')
joblib.dump(scaler,     f'{ART_DIR}/scaler.pkl')
joblib.dump(encoders,   f'{ART_DIR}/encoders.pkl')

meta = {
    'model_name'    : best_name,
    'features'      : feature_names,
    'cat_cols'      : cat_cols,
    'needs_scaling' : False,   # tree-based models do not need scaling
    'classes'       : {'0': 'Normal', '1': 'Attack'},
    'test_metrics'  : {k: float(v) for k, v in (rf_eval if best_model is tuned_rf else xgb_eval).items()}
}
with open(f'{ART_DIR}/meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('Saved to ./artifacts/:')
for f in os.listdir(ART_DIR):
    print(' ', f)

---
## 9. Discussion & Conclusion

**Cross-model findings.** All five algorithms exceed F1 = 0.94 on the held-out test set, but separation is clear: tree-based ensembles (Random Forest, XGBoost) sit at the top with F1 ≈ 0.999 and ROC-AUC = 1.000, KNN follows (F1 ≈ 0.996), SVM and Logistic Regression form the linear-baseline floor (F1 ≈ 0.95–0.97). This ordering matches the published NSL-KDD literature and confirms that *non-linear* decision surfaces are needed to capture attack patterns such as Neptune-DoS and port scans.

**Tuning effect.** GridSearchCV on Random Forest and XGBoost did not lift F1 above the already-saturated baseline — a meaningful *negative* result: it shows that the default ensembles were already well-configured for this benchmark and that the improvement budget should be spent on data quality and feature engineering rather than further hyperparameter search.

**Feature engineering value.** Both engineered features (`bytes_ratio`, `error_rate_total`) appear in the top-5 of the tuned model's feature-importance ranking, confirming that domain-informed transforms still add signal on a well-studied dataset.

**Limitations.** NSL-KDD is a *legacy* benchmark; modern encrypted traffic (TLS 1.3, QUIC) and zero-day attacks are not represented. The reported metrics therefore upper-bound, but do not directly predict, real-world IDS performance.

**Practical contribution.** The Streamlit application (`app.py`) demonstrates how the final model can be deployed: a SOC analyst pastes connection features (or uploads a flow-level CSV) and gets a probability-calibrated normal/attack verdict in real time.

**Numbered conclusions:**
1. Five ML models were trained on 100 778 NSL-KDD records; all reached F1 ≥ 0.94 on 25 195 held-out test samples.
2. The two best models — tuned Random Forest and tuned XGBoost — achieved F1 ≈ 0.999 and ROC-AUC = 1.000.
3. Two engineered features ranked in the top-5 importances of the deployment model.
4. Hyperparameter tuning confirmed default settings, demonstrating model saturation on NSL-KDD.
5. The selected model is deployed in a Streamlit web app for real-time intrusion detection.

**Future work.** Extend to multi-class (DoS / Probe / R2L / U2R), evaluate on CIC-IDS-2017/2018 to test domain transfer, add online learning for concept drift.